## **Partie traitement du dataset BindingDB**

In [ ]:
import warnings
import subprocess
import sys

# on cache temporairement les warnings de dépendances pip 
warnings.filterwarnings("ignore", category=UserWarning)
print("🚀 Mise à jour de pip et installation des packages principaux :")

# 1.PyTorch optimisé pour CUDA 12.8 (H100)
%pip install -q --upgrade pip
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
# 2.Hugging Face & Modèles Protéiques
%pip install -q transformers accelerate bitsandbytes safetensors huggingface_hub fair-esm
# 3.Chimie & Drug Discovery 
%pip install -q rdkit pubchempy py3Dmol pymol-open-source
# 4.Bio & Protéines
%pip install -q biopython biotite
# 5.Outils haute performance
%pip install -q joblib tqdm pandas numpy polars pyarrow fastparquet
# 6.SA Score (faisabilité synthèse moléculaire)
!wget -q https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/sascorer.py

import torch
import rdkit
from rdkit import Chem
import transformers
import esm

print(f"La version de PyTorch est: {torch.__version__}")
print(f"La version de RDKit est: {rdkit.__version__}")
print(f"la version de Transformers est: {transformers.__version__}")
print(f"la version de fair-esm est: {esm.__version__}")
print(f"la version de CUDA disponible est: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"le modèle de GPU est: {torch.cuda.get_device_name(0)}")
    print(f"la VRAM est de: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"la version de CUDA est: {torch.version.cuda}")
    
    # on applique une optimisations spécifiques pour H100 / Hopper
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('high')
    print("TensorFloat-32 activé : True")
else:
    print("on bascule vers le cpu si aucun gpu n'est disponible")

In [ ]:
import warnings
import subprocess
import sys

# on cache temporairement les warnings de dépendances pip 
warnings.filterwarnings("ignore", category=UserWarning)
print("Installation contrôlée de torch-geometric et extensions:")
#on utilise exactement la meme version de PyTorch
torch_version = torch.__version__.split('+')[0]   # ex: 2.4.0 ou 2.5.0
cuda_version = "cu128" if torch.cuda.is_available() else "cpu"
print(f" la version de PyTorch  est : {torch_version} | la version de CUDA détécté : {cuda_version}")
# on installe les dépendances
%pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv \
    -f https://data.pyg.org/whl/torch-{torch_version}+{cuda_version}.html
%pip install -q torch-geometric
# on installe e3nn (pour les modèles équivariants avancés)
%pip install -q e3nn

print("\n Installation de PyTorch Geometric terminée.")

import torch_geometric
import torch_scatter
import e3nn

print(f"la version de torch-geometric est: {torch_geometric.__version__}")
print(f"la version de torch-scatter est: {torch_scatter.__version__}")
print(f"la version de e3nn est: {e3nn.__version__}")

if torch.cuda.is_available():
    print("GPU disponible → PyG devrait utiliser CUDA")
else:
    print("on bascule vers le cpu si aucun gpu n'est disponible")


In [ ]:
import os
import requests

# Télécharger le fichier de scores nécessaire pour sascorer
url = "https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/fpscores.pkl.gz"
if not os.path.exists("fpscores.pkl.gz"):
    print("Téléchargement de fpscores.pkl.gz...")
    r = requests.get(url)
    with open("fpscores.pkl.gz", "wb") as f:
        f.write(r.content)
    print("Téléchargement terminé.")
else:
    print("Le fichier fpscores.pkl.gz est déjà présent.")

In [ ]:
import os
import random
import numpy as np
import torch
import gc
import warnings

# on désactive les warnings 
warnings.filterwarnings('ignore')
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
# on crée une fonction pour la reproductibilité maximale 
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # on adopte la reproductibilité stricte
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    
    print(f"la graine aléatoire est fixée à {seed}")
    
seed_everything(42)
# on choisit une configuration optimisée pour GPU H100
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    print(f"\nle GPU détecté est : {gpu_name}")
    print(f"la VRAM disponible : {vram_gb:.2f} GB")
    # on choisit une optimisations Hopper / H100
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('high')
    # on active le torch.compile pour un gain de vitesse important
    print(f"le TensorFloat-32 est actif : {torch.backends.cuda.matmul.allow_tf32}")
else:
    DEVICE = torch.device("cpu")
# on procède à un nettoyage de la mémoire
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
print("la configuration de l'environnement est terminée")

In [ ]:
import pandas as pd
import os
import pyarrow.csv as pc
import pyarrow as pa
import gc

print("🔍 Recherche du fichier BindingDB TSV...")
input_dir = "/kaggle/input"
dataset_folder = "datasets/jakeadam68/bindingdb-all-202602"
tsv_path = None
for root, dirs, files in os.walk(os.path.join(input_dir, dataset_folder)):
    for file in files:
        if file.endswith(".tsv"):
            tsv_path = os.path.join(root, file)
            break
    if tsv_path:
        break
if not tsv_path:
    raise FileNotFoundError("Aucun fichier .tsv n'a été trouvé dans le dataset BindingDB")
else:
    print(f"le fichier tsv a été trouvé au chemin spécifié : {tsv_path}")
    print(f"la taille du fichier est de : {os.path.getsize(tsv_path) / (1024**3):.2f} GB")

parquet_path = "bindingdb_full.parquet"
if os.path.exists(parquet_path):
    print(f"le fichier Parquet existe déjà, on le charge directement")
    full_df = pd.read_parquet(parquet_path)
else:
    print("📖 Lecture du fichier tsv:")
    
    try:
        # Lecture du fichier avec pyarrow.csv 
        read_options = pc.ReadOptions(use_threads=True, block_size=10<<20)  # 10MB blocks
        parse_options = pc.ParseOptions(delimiter='\t')
        convert_options = pc.ConvertOptions()
        table = pc.read_csv(
            tsv_path,
            read_options=read_options,
            parse_options=parse_options,
            convert_options=convert_options
        )
        
        full_df = table.to_pandas()
        
        print(f"\nLecture terminée : {len(full_df):,} lignes × {len(full_df.columns):,} colonnes")
        print("\n" + "="*90)
        print("📋 Voici la liste complète des colonnes du fichier bindingDB:")
        print("="*90)
        for i, col in enumerate(full_df.columns.tolist(), 1):
            print(f"   {i:3d}. {col}")
        print("\n📊 Voici un aperçu détaillé des colonnes :")
        cols_df = pd.DataFrame({
            'Colonne': full_df.columns,
            'Type': full_df.dtypes.values,
            'Exemple': [str(full_df[col].iloc[0])[:100] + "..." 
                        if len(str(full_df[col].iloc[0])) > 100 
                        else str(full_df[col].iloc[0]) 
                        for col in full_df.columns]
        })
        display(cols_df)
        
        # sauvegarde du fichier au format parquet
        print("\n💾 Sauvegarde en parquet avec compression snappy:")
        full_df.to_parquet(
            parquet_path,
            index=False,
            compression='snappy',
            engine='pyarrow'
        )
        print(f"✅ le fichier a été sauvegardé au chemin spécifié : {parquet_path} "
              f"({os.path.getsize(parquet_path)/(1024**2):.1f} MB)")
    except Exception as e:
        print(f"❌ Une erreur est survenu lors de la lecture/conversion du fichier : {e}")
        raise
        print("\n" + "="*80)
print("📊 Voici les statistiques du fichier:")
print("="*80)
print(f"le nombre total de lignes est: {len(full_df):,}")
print(f"le nombre total de colonnes est    : {len(full_df.columns)}")
print(f" la mémoire utilisée est de: {full_df.memory_usage(deep=True).sum() / (1024**3):.2f} GB")
print("\nVoici la répartition des types de données :")
print(full_df.dtypes.value_counts())
# on nettoie la mémoire
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
import pandas as pd
import numpy as np
import re
import gc

df = pd.read_parquet("/kaggle/input/datasets/jakeadam68/bindingdb-onco-admet/bindingdb_full.parquet")
print(f"le dataset initial chargé contient: {len(df):,} lignes × {len(df.columns)} colonnes")
print("\n🔎 Filtrage initial (affinité + SMILES valide):")
affinity_cols = ['Ki (nM)', 'IC50 (nM)', 'Kd (nM)', 'EC50 (nM)']
has_affinity = df[affinity_cols].notna().any(axis=1)
df = df[has_affinity & 
        df['Ligand SMILES'].notna() & 
        df['Ligand SMILES'].str.strip().ne('')].copy()
print(f"Après avoir appliquer le filtrage affinité + SMILES on obtient : {len(df):,} lignes")
# on applique le filtre pour sélectionner le génome humain
human_mask = df['Target Source Organism According to Curator or DataSource'].str.contains(
    r'human|homo sapiens', case=False, na=False, regex=True
)
df = df[human_mask].copy()
print(f"Après avoir appliquer le filtrage sur les cibles humaines on obtient: {len(df):,} lignes")
print("Extraction de la meilleure affinité et calcul de pAff:")
def get_best_affinity(row):
    for col in ['Ki (nM)', 'IC50 (nM)', 'Kd (nM)', 'EC50 (nM)']:
        val = row[col]
        if pd.notna(val):
            val_str = str(val).strip().upper()
            if val_str in ['ND', 'N.D.', 'N/A', 'NONE', 'NOT DETERMINED', '-', 'NA']:
                continue
            # Nettoyage des symboles dans les valeurs
            val_str = re.sub(r'[<>≈~]', '', val_str)
            val_str = val_str.replace(' ', '').replace(',', '')
            val_str = val_str.replace('NM', '').replace('NМ', '')
            # Gestion des intervalles de valeur d'affinité (ex: 10-50 ) avec moyenne géométrique
            if '-' in val_str:
                try:
                    parts = [float(p) for p in val_str.split('-') if p.strip()]
                    if len(parts) == 2:
                        num_val = (parts[0] * parts[1]) ** 0.5   # Moyenne géométrique
                        return num_val, col
                    elif len(parts) == 1:
                        num_val = parts[0]
                except:
                    continue

            try:
                num_val = float(val_str)
                if 0 < num_val <= 5_000_000:
                    return num_val, col
            except:
                continue
    return np.nan, None
    
temp = df[affinity_cols].apply(get_best_affinity, axis=1, result_type='expand')
df[['affinity_nM', 'aff_type']] = temp
df['affinity_nM'] = pd.to_numeric(df['affinity_nM'], errors='coerce')
df = df[df['affinity_nM'] > 0].copy()   # On garde seulement les valeurs positives
# on calcule la puissance d'affinité et on prend les puissances d'affinités entre 3 et 11
df['pAff'] = -np.log10(df['affinity_nM'] * 1e-9)
# Nettoyage mémoire immédiat
del temp
gc.collect()
# on extrait les informations des séquences protéiques
seq_col = 'BindingDB Target Chain Sequence 1'
uniprot_col = 'UniProt (SwissProt) Primary ID of Target Chain 1'
name_col = 'UniProt (SwissProt) Recommended Name of Target Chain 1'
df = df.dropna(subset=[seq_col, uniprot_col]).copy()
df = df.rename(columns={
    seq_col: 'protein_sequence',
    uniprot_col: 'uniprot_id',
    name_col: 'protein_name'
})
# on ne garde que les colonnes pertinentes pour la suite
keep_cols = [
    'BindingDB Reactant_set_id',
    'Ligand SMILES',
    'BindingDB Ligand Name',
    'affinity_nM',
    'aff_type',
    'pAff',
    'protein_sequence',
    'uniprot_id',
    'protein_name',
    'Target Name',
    'PDB ID(s) for Ligand-Target Complex',
    'PDB ID(s) of Target Chain 1',
    'Ligand HET ID in PDB',
    'Number of Protein Chains in Target (>1 implies a multichain complex)',
    'DrugBank ID of Ligand',
    'ChEMBL ID of Ligand',
    'PubChem CID',
    # Colonnes pour traiter les cas ou les séquences protéiques sont isoformes 
    'UniProt (SwissProt) Secondary ID(s) of Target Chain 1',
    'UniProt (TrEMBL) Primary ID of Target Chain 1',
    'UniProt (TrEMBL) Secondary ID(s) of Target Chain 1'
]
print("📊 Début de l'agrégation séléctive:")
# Définir la hiérarchie des mesures (Ki > Kd > IC50 > EC50)
priority_map = {'Ki (nM)': 4, 'Kd (nM)': 3, 'IC50 (nM)': 2, 'EC50 (nM)': 1}
df['priority'] = df['aff_type'].map(priority_map).fillna(0)
# Identifier le meilleur type disponible pour chaque paire (Ligand-Protéine)
df['max_priority'] = df.groupby(['Ligand SMILES', 'uniprot_id'])['priority'].transform('max')
# Filtrage pour ne garder que le meilleur type identifié
df_best = df[df['priority'] == df['max_priority']].copy()
# Construction DYNAMIQUE du dictionnaire d'agrégation à partir de keep_cols
# on vérifie quelles colonnes demandées sont réellement présentes dans le DataFrame
available_cols = [col for col in keep_cols if col in df_best.columns]
# on crée le dictionnaire : par défaut 'first' pour tout le monde
agg_dict = {col: 'first' for col in available_cols}
# sauf pour les colonnes numériques d'affinité où on impose la MÉDIANE (pour le lissage)
if 'pAff' in agg_dict: agg_dict['pAff'] = 'median'
if 'affinity_nM' in agg_dict: agg_dict['affinity_nM'] = 'median'
# on retire les colonnes de groupement du dictionnaire pour éviter les erreurs Pandas
for col in ['Ligand SMILES', 'uniprot_id']:
    if col in agg_dict: del agg_dict[col]
# Exécution de l'agrégation finale
df_clean = df_best.groupby(['Ligand SMILES', 'uniprot_id']).agg(agg_dict).reset_index()
# Filtrage pAff après aggrégation
df_clean = df_clean[(df_clean['pAff'] >= 3.0) & (df_clean['pAff'] <= 11.0)].copy()
print(f"\nle dataset final nettoyé contient : {len(df_clean):,} paires uniques")
print(f"pAff → min: {df_clean['pAff'].min():.2f} | max: {df_clean['pAff'].max():.2f} | moyenne: {df_clean['pAff'].mean():.2f}")
# Sauvegarde du fichier au format csv et parquet
df_clean.to_parquet("bindingdb_human_clean.parquet", index=False, compression='snappy')
df_clean.to_csv("bindingdb_human_clean.csv", index=False)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Chargement du dataset BindingDB
df = pd.read_parquet("bindingdb_human_clean.parquet")

# Configuration esthétique
plt.style.use('default')
sns.set_palette("viridis")
fig = plt.figure(figsize=(20, 14))
plt.suptitle(f"Statistiques Descriptives du Dataset Diamond Humain ({len(df):,} paires)", 
             fontsize=22, fontweight='bold', y=0.95)

# Distribution du pAff 
ax1 = plt.subplot(2, 2, 1)
sns.histplot(df['pAff'], bins=50, kde=True, color='royalblue', ax=ax1)
ax1.set_title("Distribution de la Puissance d'Affinité (pAff)", fontsize=16, fontweight='bold')
ax1.set_xlabel("pAff (-log10[M])", fontsize=12)
ax1.axvline(df['pAff'].mean(), color='red', linestyle='--', label=f"Moyenne: {df['pAff'].mean():.2f}")
ax1.axvline(df['pAff'].median(), color='green', linestyle='-', label=f"Médiane: {df['pAff'].median():.2f}")
ax1.legend()

# Répartition des Types de Mesures
ax2 = plt.subplot(2, 2, 2)
type_counts = df['aff_type'].value_counts()
sns.barplot(x=type_counts.index, y=type_counts.values, palette="magma", ax=ax2)
ax2.set_title("Répartition des Types de Mesures", fontsize=16, fontweight='bold')
ax2.set_ylabel("Nombre de paires", fontsize=12)
# Ajout des pourcentages sur les barres
for i, v in enumerate(type_counts.values):
    ax2.text(i, v + 20000, f"{(v/len(df)*100):.1f}%", ha='center', fontweight='bold')

# Boxplot pAff par Type de Mesure 
ax4 = plt.subplot(2, 2, 3)
sns.boxplot(x='aff_type', y='pAff', data=df, palette="viridis", ax=ax4)
ax4.set_title("Variabilité du pAff par Type de Mesure", fontsize=16, fontweight='bold')
ax4.set_xlabel("Type d'affinité", fontsize=12)

plt.tight_layout(rect=[0, 0.03, 1, 0.92])
plt.savefig("pAff_human_statistics.png", dpi=300, bbox_inches='tight')
plt.show()

# Affichage des statistiques numériques clés
print(f"\n Résumé des statistiques numériques:")
print(f"Nombre total de ligands uniques: {df['Ligand SMILES'].nunique():,}")
print(f"\nStatistiques pAff :")
print(df['pAff'].describe().round(3))

In [ ]:
import pandas as pd

cosmic_path = "/kaggle/input/datasets/rayanchakibidris/cosmic-genomescreen-v103/Cosmic_CancerGeneCensus_Tsv_v103_GRCh38/Cosmic_CancerGeneCensus_v103_GRCh38.tsv/Cosmic_CancerGeneCensus_v103_GRCh38.tsv"
# Lecture du fichier au format tsv
df_cosmic = pd.read_csv(cosmic_path, sep='\t', low_memory=False)
print(f"le fichier est chargé avec succès et contient: {len(df_cosmic):,} lignes × {len(df_cosmic.columns)} colonnes\n")
print("📋 Colonnes présentes dans le fichier Cosmic Cancer Gene Census:")
for i, col in enumerate(df_cosmic.columns.tolist(), 1):
    print(f"   {i:2d}. {col}")
print("\n" + "="*80)
print("Voici un aperçu des 5 premières lignes :")
display(df_cosmic.head())
# on affiche les statistiques du fichier 
print("\nTypes de données :")
print(df_cosmic.dtypes.value_counts())

In [ ]:
import pandas as pd

# Remplace par le vrai nom de ton fichier
df_oncokb = pd.read_csv("/kaggle/input/datasets/rayanchakibidris/oncokb-dataset/cancerGeneList.tsv", sep='\t')
print("Colonnes du fichier OncoKB :")
print(df_oncokb.columns.tolist())
print("\nAperçu des 5 premières lignes :")
display(df_oncokb.head())

In [ ]:
import pandas as pd

intogen_path = "/kaggle/input/datasets/rayanchakibidris/intogen/2024-06-18_IntOGen-Drivers/Unfiltered_drivers.tsv"
print("📂 Chargement du fichier IntOGen unfiltered_drivers.tsv:")
# Lecture du fichier IntOGen - Unfiltered.csv 
df_intogen = pd.read_csv(intogen_path, sep='\t', low_memory=False)
print(f"le fichier a été chargé avec succès et contient : {len(df_intogen):,} lignes × {len(df_intogen.columns)} colonnes\n")
# Affichage des colonnes
print("📋 Colonnes présentes dans le fichier IntOGen:")
for i, col in enumerate(df_intogen.columns.tolist(), 1):
    print(f"   {i:2d}. {col}")
print("\n" + "="*90)
print("Voici un aperçu des 5 premières lignes:")
display(df_intogen.head())
print("\nTypes de données:")
print(df_intogen.dtypes.value_counts())
# Statistiques rapides sur les colonnes importantes
if 'ROLE' in df_intogen.columns:
    print("\nDistribution des rôles (Oncogene / TSG / etc.) :")
    print(df_intogen['ROLE'].value_counts())
if 'CANCER_TYPE' in df_intogen.columns:
    print(f"\nNombre de types de cancer uniques : {df_intogen['CANCER_TYPE'].nunique()}")

In [ ]:
import pandas as pd

compendium_path = "/kaggle/input/datasets/rayanchakibidris/intogen/2024-06-18_IntOGen-Drivers/Compendium_Cancer_Genes.tsv"
print("📂 Chargement du fichier IntOGen Compendium_Cancer_Genes.tsv:")
# Lecture du fichier IntOGen - Compendium_Cancer_Genes.tsv
df_compendium = pd.read_csv(compendium_path, sep='\t', low_memory=False)
print(f"le fichier a été chargé avec succès et contient : {len(df_compendium):,} lignes × {len(df_compendium.columns)} colonnes\n")
# Affichage des colonnes
print("📋 Colonnes présentes dans Compendium_Cancer_Genes.tsv :")
for i, col in enumerate(df_compendium.columns.tolist(), 1):
    print(f"   {i:2d}. {col}")
print("\n" + "="*100)
print("Voici un aperçu des 5 premières lignes :")
display(df_compendium.head())
# Statistiques générales
print("\nTypes de données :")
print(df_compendium.dtypes.value_counts())
# Analyse des colonnes importantes
important_cols = ['GENE', 'ROLE', 'CANCER_TYPE', 'TIER', 'DRIVER', 'ONCOGENE', 'TSG']
for col in important_cols:
    if col in df_compendium.columns:
        if df_compendium[col].dtype == 'object':
            print(f"\nDistribution de la colonne '{col}' :")
            print(df_compendium[col].value_counts().head(10))
        else:
            print(f"\nStatistiques de '{col}' :")
            print(df_compendium[col].describe())
# Nombre de gènes uniques
if 'GENE' in df_compendium.columns:
    print(f"\nNombre de gènes uniques dans le Compendium : {df_compendium['GENE'].nunique():,}")
# Sauvegarde temporaire pour usage futur
df_compendium.to_parquet("intogen_compendium_cancer_genes.parquet", index=False)
print("\n💾 Fichier sauvegardé en parquet : intogen_compendium_cancer_genes.parquet")

In [ ]:
import pandas as pd
import gc
from rdkit import Chem
from rdkit.Chem import Descriptors, QED
from tqdm.auto import tqdm
import sascorer 

# on charge les chemins des fichiers tsv
df_cosmic_census = pd.read_csv("/kaggle/input/datasets/rayanchakibidris/cosmic-genomescreen-v103/Cosmic_CancerGeneCensus_Tsv_v103_GRCh38/Cosmic_CancerGeneCensus_v103_GRCh38.tsv/Cosmic_CancerGeneCensus_v103_GRCh38.tsv", sep='\t', low_memory=False)
df_cosmic_mutant = pd.read_csv("/kaggle/input/datasets/rayanchakibidris/cosmic-genomescreen-v103/Cosmic_GenomeScreensMutant_Tsv_v103_GRCh38/Cosmic_GenomeScreensMutant_v103_GRCh38.tsv/Cosmic_GenomeScreensMutant_v103_GRCh38.tsv", sep='\t', low_memory=False)
df_oncokb = pd.read_csv("/kaggle/input/datasets/rayanchakibidris/oncokb-dataset/cancerGeneList.tsv", sep='\t', low_memory=False)
df_intogen = pd.read_csv("/kaggle/input/datasets/rayanchakibidris/intogen/2024-06-18_IntOGen-Drivers/Compendium_Cancer_Genes.tsv", sep='\t', low_memory=False)
cosmic_genes_census = set(df_cosmic_census['GENE_SYMBOL'].astype(str).str.strip().dropna())
cosmic_genes_mutant = set(df_cosmic_mutant['GENE_SYMBOL'].astype(str).str.strip().dropna())
oncokb_genes = set(df_oncokb['Hugo Symbol'].astype(str).str.strip().dropna())
intogen_genes = set(df_intogen['SYMBOL'].astype(str).str.strip().dropna())

# on analyse les chevauchements des gènes dans les différents datasets
print("\n🔍 ANALYSE DES CHEVAUCHEMENTS ENTRE SOURCES :")
print(f"le fichier COSMIC Cancer Gene Census contient: {len(cosmic_genes_census):,} gènes")
print(f"le fichier COSMIC Genome Screen Mutant contient: {len(cosmic_genes_mutant):,} gènes")
print(f"le fichier OncoKB contient: {len(oncokb_genes):,} gènes")
print(f"le fichier IntOGen Compendium contient: {len(intogen_genes):,} gènes")
print(f"\n COSMIC Cancer Gene Census ∩ COSMIC Genome Screen Mutant ∩ OncoKB : {len(cosmic_genes_census & cosmic_genes_mutant & oncokb_genes):,} gènes")
print(f"COSMIC Cancer Gene Census ∩ COSMIC Genome Screen Mutant ∩ IntOGen: {len(cosmic_genes_census & cosmic_genes_mutant & intogen_genes):,} gènes")
print(f"COSMIC Genome Screen Mutant ∩ OncoKB ∩ IntOGen: {len(cosmic_genes_mutant & oncokb_genes & intogen_genes):,} gènes")
print(f"COSMIC Cancer Gene Census ∩ Cancer Genome Screen Mutant ∩ OncoKB ∩ IntOGen: {len(cosmic_genes_census & cosmic_genes_mutant & oncokb_genes & intogen_genes):,} gènes")
elite_driver_genes = cosmic_genes_census.union(oncokb_genes).union(intogen_genes)
global_proteome_universe = cosmic_genes_mutant
print(f"\nl'union finale des datasets contient pour ne sélectionner que les gènes oncogénic: {len(elite_driver_genes):,} gènes oncogénic")
print(f"\nle nombre de gènes dans l'univers global est : {len(global_proteome_universe)} gènes")
# on comptes le nombre de gène unique à chaque source
only_cosmic_census = cosmic_genes_census - oncokb_genes - intogen_genes - cosmic_genes_mutant
only_oncokb = oncokb_genes - cosmic_genes_census - intogen_genes - cosmic_genes_mutant
only_intogen = intogen_genes - cosmic_genes_census - oncokb_genes - cosmic_genes_mutant 
only_cosmic_mutant = cosmic_genes_mutant - cosmic_genes_census - oncokb_genes - intogen_genes
print("les gènes uniques à chaque source :")
print(f"le nombre de gène présent uniquement dans COSMIC Cancer Gene Census est: {len(only_cosmic_census):,} gènes")
print(f"le nombre de gène présent uniquement dans COSMIC Genome Screen Mutant est: {len(only_cosmic_mutant):,} gènes")
print(f"le nombre de gène présent uniquement dans OncoKB est: {len(only_oncokb):,} gènes")
print(f"le nombre de gène présent uniquement dans IntOGen est: {len(only_intogen):,} gènes")
# Exemples de gènes uniques
if len(only_oncokb) > 0:
    print(f"\nExemples de gènes uniques à OncoKB : {list(only_oncokb)[:10]}")
if len(only_intogen) > 0:
    print(f"Exemples de gènes uniques à IntOGen : {list(only_intogen)[:10]}")
if len(only_cosmic_mutant) > 0:
    print(f"Exemples de gènes uniques à la liste extra : {list(only_cosmic_mutant)[:10]}")
# on charge le fichier BindingDB et on procède au mapping
print("\n📂 Chargement du dataset BindingDB:")
df = pd.read_parquet("bindingdb_human_clean.parquet")
print(f"Dataset BindingDB chargé : {len(df):,} paires")
print("🔗 Création du mapping UniProt → HGNC symbol:")
mapping = {}
df_hgnc = pd.read_csv("/kaggle/input/datasets/jakeadam68/hgnc-complete-set/hgnc_complete_set.txt", 
                      sep='\t', low_memory=False)
for _, row in df_hgnc.iterrows():
    symbol = row.get('symbol')
    uniprot_ids = row.get('uniprot_ids')
    if pd.notna(symbol) and pd.notna(uniprot_ids):
        symbol = str(symbol).strip()
        for uid in str(uniprot_ids).split('|'):
            uid = uid.strip()
            if uid:
                mapping[uid] = symbol
print(f"le mapping a été créé on a: {len(mapping):,} correspondances UniProt → Symbol")
# Diagnostic détaillé du mapping
print("\n📊 DIAGNOSTIC DU MAPPING UniProt → Gene Symbol :")
total_uniprot = df['uniprot_id'].nunique()
total_pairs = len(df)
mapped_pairs = df['uniprot_id'].map(mapping).notna().sum()
mapped_uniprot_unique = df['uniprot_id'].map(mapping).dropna().nunique()
print(f"Nombre total d'uniprot_id uniques dans BindingDB : {total_uniprot:,}")
print(f"Nombre d'uniprot_id uniques mappés : {mapped_uniprot_unique:,} "
      f"({mapped_uniprot_unique / total_uniprot * 100:.2f}%)")
print(f"Nombre de paires mappées                          : {mapped_pairs:,} "
      f"({mapped_pairs / total_pairs * 100:.2f}%)")
# Calcul du taux d'échec de mapping
unmapped_uniprot = total_uniprot - mapped_uniprot_unique
print(f"Nombre d'uniprot_id non mappés: {unmapped_uniprot:,} avec un taux de: {unmapped_uniprot / total_uniprot * 100:.2f}%")
# Application du filtrage multi-sources
df['gene_symbol'] = df['uniprot_id'].map(mapping)
df_final_pool = df[df['gene_symbol'].isin(global_proteome_universe)].copy()
df_final_pool['is_oncogenic'] = df_final_pool['gene_symbol'].isin(elite_driver_genes)
print(f"le nombre total de paires conservées est : {len(df_final_pool):,}")
print(f"le nombre de paires 'Drivers' (is_oncogenic=True) est : {df_final_pool['is_oncogenic'].sum():,} paires")
print(f"le nombre de paires 'Global' (is_oncogenic=False) est : {len(df_final_pool) - df_final_pool['is_oncogenic'].sum():,} paires")
print(f"le taux de conservation après filtrage oncologique est de: {len(df_final_pool)/len(df)*100:.2f}%")
# Apllication du filtrage ADMET 
def is_drug_like_quality(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return False
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        tpsa = Descriptors.TPSA(mol)
        rotb = Descriptors.NumRotatableBonds(mol)
        hba = Descriptors.NumHAcceptors(mol)
        hbd = Descriptors.NumHDonors(mol)
        qed = QED.qed(mol)
        sa_score = sascorer.calculateScore(mol)
        # Critères équilibrés et de qualité pour oncologie
        return (
            160 < mw < 950 and           # MW (poids moléculaire) : large mais raisonnable
            -7.5 < logp < 8.5 and        # LogP : tolérant pour les inhibiteurs
            tpsa < 170 and               # Polarité
            rotb <= 13 and               # Flexibilité
            hba <= 11 and                # Hydrogene acceptor
            hbd <= 5 and                 # Hydrogene Donor
            qed > 0.17 and               # QED de qualité (pas trop bas)
            sa_score < 6.8               # Synthetic accessibility raisonnable
        )
    except:
        return False
tqdm.pandas(desc="ADMET Quality")
df_final_pool['IsDrugLike'] = df_final_pool['Ligand SMILES'].progress_apply(is_drug_like_quality)
df_sota = df_final_pool[df_final_pool['IsDrugLike']].drop(columns=['IsDrugLike']).copy()
print(f"Après avoir appliquer le filtre ADMET de qualité on obtient : {len(df_sota):,} paires avec un taux de: {len(df_sota)/len(df_final_pool)*100:.2f}% des oncologiques")
# Sauvegarde du fichier au format parquet
df_sota.to_parquet("bindingdb_onco_multi_sources_final.parquet", index=False, compression='snappy')
gc.collect()

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import FilterCatalog
from tqdm.auto import tqdm
# 1. Chargement du dataset BindingDB ADMET-filtré
file_path = "bindingdb_onco_multi_sources_final.parquet"
df = pd.read_parquet(file_path)
print(f"Dataset chargé : {len(df):,} paires")
# 2. Configuration du catalogue PAINS (RDKit)
params = FilterCatalog.FilterCatalogParams()
params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
catalog = FilterCatalog.FilterCatalog(params)
# on définit une fonction qui vérifie si la molécule est Saine ( pas un PAINS)
def check_pains(smiles):
    """Renvoie True si la molécule est SAINE (pas un PAINS), False sinon."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return False
        # Si catalog.HasMatch est vrai, c'est un PAINS, donc on renvoie False
        return not catalog.HasMatch(mol)
    except:
        return False
# 3. on filtre uniquement les SMILES uniques
unique_smiles = df['Ligand SMILES'].unique()
print(f"Analyse de {len(unique_smiles):,} structures uniques pour les motifs PAINS")
# on crée un dictionnaire de mapping pour ne calculer chaque SMILES qu'une seule fois
pains_results = {}
for s in tqdm(unique_smiles, desc="Recherche motifs PAINS"):
    pains_results[s] = check_pains(s)
# 4. Application du filtre au DataFrame
df['is_clean_pains'] = df['Ligand SMILES'].map(pains_results)
df_final_diamond = df[df['is_clean_pains']].drop(columns=['is_clean_pains']).copy()
# 5. Affichage des statistiques finales
n_eliminated = len(df) - len(df_final_diamond)
print(f"\nRÉSULTATS DU FILTRAGE PAINS:")
print(f"le nombre de paires originales avant filtrage PAINS: {len(df):,}")
print(f"le nombre de paires éliminées considérée comme PAINS: {n_eliminated:,}")
print(f" le nombre de paires finales après filtrage PAINS: {len(df_final_diamond):,}")
print(f"le taux de pureté chimique: {(len(df_final_diamond)/len(df))*100:.2f}%")
# 6. Sauvegarde du dataset après filtrage PAINS
output_name = "bindingdb_onco_diamond_final.parquet"
df_final_diamond.to_parquet(output_name)
print(f"\nle dataset final est sauvegardé au chemin spécifié: {output_name}")

In [ ]:
import pandas as pd
import os

parquet_file = "/kaggle/input/datasets/jakeadam68/bindingdb-onco-admet/bindingdb_onco_diamond_final.parquet" 
# on vérifie si le chemin du fichier existe
if not os.path.exists(parquet_file):
    print(f"le fichier n'a pas été trouvé au chemin spécifié : {parquet_file}")
    # on affiche la liste des fichiers dans le répertoire courant
    !ls -lh *.parquet
else:
    print(f"le fichier a été trouvé au chemin spécifié : {parquet_file}")
    print(f"la taille du fichier est de : {os.path.getsize(parquet_file) / 1024**2:.1f} Mo\n")
    # on charge le fichier 
    try:
        # on lit uniquement les colonnes et on limite l'affichage à 10 lignes
        df_preview = pd.read_parquet(parquet_file, columns=None)  
        print(df_preview.columns.tolist())
        print(f"le nombre total de colonnes est : {len(df_preview.columns)}\n")
        # on affiche le type de donnée par colonne
        print(df_preview.dtypes)
        print("\n")
        # on affiche un apercu des dix premières lignes
        pd.set_option('display.max_columns', None)  # on affiche toutes les colonnes
        print(df_preview.head(10))
        print("\n")
        # on affiche les statistiques des colonnes
        print(df_preview.describe())
        print("\n")
        # on affiche le taux de valeurs manquantes par colonne
        missing = df_preview.isna().sum()
        missing_percent = (missing / len(df_preview)) * 100
        missing_df = pd.DataFrame({
            'Valeurs manquantes': missing,
            '% manquant': missing_percent.round(2)
        })
        print(missing_df[missing_df['Valeurs manquantes'] > 0])
        print("\n")
        # on affiche le résultat des colonnes spécifiques qui nous intéresse
        interesting_cols = ['Ligand SMILES', 'Target Name', 'pAff', 'uniprot_id', 'protein_sequence', 'protein_name']
        for col in interesting_cols:
            if col in df_preview.columns:
                print(f"le type de colonne '{col}' est : {df_preview[col].dtype}")
                print(f"le nombre de valeurs uniques : {df_preview[col].nunique()}")
                print(f"Voici un exemple: {df_preview[col].iloc[0]}")
                print(f"le nombre de valeurs NaN est: {df_preview[col].isna().sum()}")
                print("")
        # on réinitialise l'option d'affichage
        pd.reset_option('display.max_columns')
        
    except Exception as e:
        print(f"Erreur lors de la lecture : {e}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Descriptors, QED
import sascorer
from tqdm.auto import tqdm

# Configuration esthétique des graphiques
plt.style.use('default')
sns.set_palette("husl")
sns.set_context("notebook", font_scale=1.2)
# Chargement du dataset final
df_final = pd.read_parquet("/kaggle/input/datasets/jakeadam68/bindingdb-onco-admet/bindingdb_onco_diamond_final.parquet")
print(f"le dataset final a été chargé et contient: {len(df_final):,} paires")
# Affichage des statistiques 
print(f"le nombre total de paires est: {len(df_final):,}")
print(f"le nombre de gènes uniques est: {df_final['gene_symbol'].nunique():,}")
print(f"le nombre de ligands uniques est: {df_final['Ligand SMILES'].nunique():,}")
print(f"le nombre de protéines uniques est: {df_final['uniprot_id'].nunique():,}")
print(f"\nVoici les statistiques sur la puissance d'affinité (pAff):")
print(df_final['pAff'].describe().round(3))
# Calcul des descripteurs moléculaires 
print("\nCalcul des descripteurs moléculaires:")
unique_smiles = df_final['Ligand SMILES'].unique()
mw_list = []
logp_list = []
qed_list = []
sa_list = []
tpsa_list = []
for smiles in tqdm(unique_smiles, desc="Calcul descripteurs"):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            continue
        # 1. Calculer tout dans des variables temporaires
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        qed_val = QED.qed(mol)
        sa = sascorer.calculateScore(mol)
        tpsa = Descriptors.TPSA(mol)
        
        mw_list.append(mw)
        logp_list.append(logp)
        qed_list.append(qed_val)
        sa_list.append(sa)
        tpsa_list.append(tpsa)
        
    except Exception as e:
        continue
# Vérification des longueurs avant de créer le DataFrame
print(f"le nombre de molécules analysées avec succès est: {len(mw_list)}")
# Création sécurisée du DataFrame
props_df = pd.DataFrame({
    'Molecular Weight (Da)': mw_list,
    'LogP': logp_list,
    'QED': qed_list,
    'SA Score': sa_list,
    'TPSA': tpsa_list
})
# Visualisations des figures
fig = plt.figure(figsize=(20, 16))
# 1. Distribution pAff
ax1 = plt.subplot(2, 3, 1)
sns.histplot(df_final['pAff'], bins=50, kde=True, color='blue', ax=ax1)
ax1.set_title('Distribution de pAff', fontsize=14, fontweight='bold')
ax1.set_xlabel('pAff (-log10(Ki))')
# 2. Molecular Weight
ax2 = plt.subplot(2, 3, 2)
sns.histplot(props_df['Molecular Weight (Da)'], bins=50, kde=True, color='green', ax=ax2)
ax2.set_title('Distribution du Poids Moléculaire', fontsize=14, fontweight='bold')
# 3. LogP
ax3 = plt.subplot(2, 3, 3)
sns.histplot(props_df['LogP'], bins=50, kde=True, color='orange', ax=ax3)
ax3.set_title('Distribution du LogP', fontsize=14, fontweight='bold')
# 4. QED
ax4 = plt.subplot(2, 3, 4)
sns.histplot(props_df['QED'], bins=50, kde=True, color='purple', ax=ax4)
ax4.set_title('Distribution du QED Score', fontsize=14, fontweight='bold')
# 5. SA Score
ax5 = plt.subplot(2, 3, 5)
sns.histplot(props_df['SA Score'], bins=50, kde=True, color='red', ax=ax5)
ax5.set_title('Distribution du SA Score', fontsize=14, fontweight='bold')
# 6. Boxplot comparatif
ax6 = plt.subplot(2, 3, 6)
sns.boxplot(data=props_df, ax=ax6)
ax6.set_title('Boxplots des Descripteurs Moléculaires', fontsize=14, fontweight='bold')
ax6.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig("dataset_final_statistics_visualization.png", dpi=300, bbox_inches='tight')
# Affichage des statistiques numériques
print("\nStatistiques numériques des descripteurs moléculaires:")
print(props_df.describe().round(3))
# on affiche le top 15 des gènes les plus représentés
print("\nVoici le top 15 des gènes les plus représentés:")
print(df_final['gene_symbol'].value_counts().head(15))
print("\nStatistiques visuels des descripteurs moléculaires:")
# Sauvegarde du fichier au format csv
props_df.describe().round(3).to_csv("dataset_final_molecular_properties_stats.csv")

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt

# on charge le fichier BindingDB
df = pd.read_parquet("/kaggle/input/datasets/jakeadam68/bindingdb-onco-admet/bindingdb_onco_diamond_final.parquet")
print(f"Avant de nettoyer les séquences protéiques on avait : {len(df):,} lignes")
valid_aa_pattern = re.compile(r'^[ACDEFGHIKLMNPQRSTVWY]+$') # on définit le motif regex compilé pour les 20 lettres des acides aminés standard 

def is_valid_protein(seq):
    if not isinstance(seq, str):
        return False
    seq_clean = seq.strip().upper()
    if len(seq_clean) < 50:   # on ne garde uniquement que les séquences protéiques qui ont une longueur > 50
        return False
    return bool(valid_aa_pattern.match(seq_clean))
valid_mask = df['protein_sequence'].apply(is_valid_protein)
df_clean = df[valid_mask].copy()
df_clean['protein_sequence'] = df_clean['protein_sequence'].str.strip().str.upper()
print(f"Après avoir nettoyer les séquences protéiques on obtient : {len(df_clean):,} lignes")
print(f"le nombre de lignes supprimées sont : {len(df) - len(df_clean):,} avec un taux de : {(len(df) - len(df_clean))/len(df)*100:.3f} %")
print("la longueur minimale de la séquence protéique est :", df_clean['protein_sequence'].str.len().min())
print("la longueur moyenne de la séquence protéique est :", df_clean['protein_sequence'].str.len().mean().round(0))
print("la longueur maximale de la séquence protéique est :", df_clean['protein_sequence'].str.len().max())
# Diagnostic de vérification 
has_invalide = df_clean['protein_sequence'].str.contains(r'[^ACDEFGHIKLMNPQRSTVWY]', regex=True).any()
print(f"Vérification des caractères invalides restants : {has_invalide}") 
print("les valeurs non-numérique restantes :", df_clean['protein_sequence'].isna().any())
# voici la visualisation des statistiques obtenues
lengths = df_clean['protein_sequence'].apply(lambda seq: len(seq.strip()))
print(lengths.describe())
plt.hist(lengths, bins=50)
plt.title('Distribution des longueurs de séquences protéiques')
plt.xlabel('la longueur (AA)')
plt.ylabel('le nombre de séquences')
plt.savefig("protein_sequence_length_visualization.png", dpi=300, bbox_inches='tight')
plt.show()
# on sauvegarde le fichier sous format .parquet
df_clean.to_parquet("bindingdb_onco_diamond_clean.parquet", index=False)

In [ ]:
# on charge le fichier BindingDB
df_clean = pd.read_parquet("bindingdb_onco_diamond_clean.parquet")
# on affiche un apercu des cibles et leurs nombres
print("le top 15 des cibles après nettoyage :")
print(df_clean['Target Name'].value_counts().head(15))
# on affiche la puissance d'affinité minimale, maximale, moyenne 
print(f"\n le pAff min/max/moyen : {df_clean['pAff'].min():.2f} / {df_clean['pAff'].max():.2f} / {df_clean['pAff'].mean():.2f}")

In [ ]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import Align
from io import StringIO
from Bio import SeqIO
from tqdm.auto import tqdm
import time
import warnings
warnings.filterwarnings("ignore")
# on charge le fichier BindingDB nettoyé
df = pd.read_parquet("bindingdb_onco_diamond_clean.parquet")
print(f"le dataset a été chargé et contient : {len(df):,} paires et {df['uniprot_id'].nunique():,} protéines uniques")
# on fait un alignement Smith-Waterman
aligner = Align.PairwiseAligner()
aligner.mode = 'local'
aligner.open_gap_score = -0.5
aligner.extend_gap_score = -0.1
# on crée une fonction pour récupérer la séquence canonique et celles des isoformes officielles
def fetch_uniprot_family(base_ids):
    """on récupère la séquence canonique + toutes les isoformes officielles"""
    all_base = set()
    for uid in base_ids:
        if pd.notna(uid):
            parts = str(uid).replace(',', ';').split(';')
            for p in parts:
                all_base.add(p.strip().split('-')[0])
    combined_fasta = ""
    for bid in all_base:
        url = f"https://rest.uniprot.org/uniprotkb/search?query=accession:{bid}&format=fasta&includeIsoform=true"
        try:
            r = requests.get(url, timeout=15)
            if r.status_code == 200:
                combined_fasta += r.text
        except:
            continue
        time.sleep(0.1)
    if not combined_fasta:
        return []
    return list(SeqIO.parse(StringIO(combined_fasta), "fasta"))
results = []
print(f"\nAnalyse rigoureuse sur {df['uniprot_id'].nunique():,} protéines:")
for _, row in tqdm(df.drop_duplicates(subset=['uniprot_id']).iterrows(), total=df['uniprot_id'].nunique()):
    candidate_ids = [row['uniprot_id']]
    for col in ['UniProt (SwissProt) Secondary ID(s) of Target Chain 1',
                'UniProt (TrEMBL) Primary ID of Target Chain 1',
                'UniProt (TrEMBL) Secondary ID(s) of Target Chain 1']:
        if col in row and pd.notna(row[col]):
            candidate_ids.extend(str(row[col]).split(';'))
    isoforms = fetch_uniprot_family(candidate_ids)
    if not isoforms:
        results.append({
            'gene': row['gene_symbol'],
            'uniprot_primary': row['uniprot_id'],
            'best_match': 'N/A',
            'identity': 0.0,
            'length_diff': 0,
            'status': 'Erreur API',
            'confidence': 0.0
        })
        continue
    seq_dataset = str(row['protein_sequence']).strip()
    best_score = -1
    best_id = ""
    canonical_len = 0
    for isoform in isoforms:
        if '-1' in isoform.id or '-' not in isoform.id:
            canonical_len = len(isoform.seq)
        score = aligner.score(isoform.seq, seq_dataset)
        if score > best_score:
            best_score = score
            best_id = isoform.id
    identity = (best_score / len(seq_dataset)) * 100 if len(seq_dataset) > 0 else 0.0
    length_diff = abs(len(seq_dataset) - canonical_len)
    # Classification rigoureuse à plusieurs niveaux
    if length_diff <= 5 and identity >= 99.0:
        status = "Canonique (haute confiance)"
        confidence = 0.95
    elif '-' in best_id and identity >= 98.0:
        status = "Isoforme explicite"
        confidence = 0.90
    elif identity >= 97.0 and len(seq_dataset) < 0.92 * canonical_len:
        status = "Fragment de canonique"
        confidence = 0.75
    elif identity >= 95.0:
        status = "Canonique avec variation mineure"
        confidence = 0.80
    else:
        status = "Ambigu / Isoforme non standard"
        confidence = 0.50
    results.append({
        'gene': row['gene_symbol'],
        'uniprot_primary': row['uniprot_id'],
        'best_match_uniprot': best_id,
        'identity_score_%': round(identity, 2),
        'length_diff': length_diff,
        'status': status,
        'confidence': confidence
    })


df_sota = pd.DataFrame(results)

# Configuration du style général
sns.set_theme(style="white", palette="muted")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 10))
plt.subplots_adjust(wspace=0.3)

# Distribution du Statut Biologique
# On trie par fréquence pour un meilleur rendu visuel
status_order = df_sota['status'].value_counts().index
sns.countplot(data=df_sota, y='status', order=status_order, palette="viridis", ax=ax1, hue='status', legend=False)

# Ajout des pourcentages sur les barres
total = len(df_sota)
for i, p in enumerate(ax1.patches):
    percentage = f'{100 * p.get_width() / total:.1f}%'
    x = p.get_width() + 5
    y = p.get_y() + p.get_height() / 2
    ax1.annotate(f"{int(p.get_width())} ({percentage})", (x, y), fontsize=12, fontweight='bold', va='center')

ax1.set_title("A. Classification de l'Identité Biologique (N=673)", fontsize=18, fontweight='bold', pad=20)
ax1.set_xlabel("Nombre de protéines uniques", fontsize=14)
ax1.set_ylabel("")
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Analyse de la Fiabilité (Confidence Score)
# Utilisation d'un camembert "Donut" pour le score de confiance
conf_counts = df_sota['confidence'].value_counts().sort_index(ascending=False)
colors = sns.color_palette("flare", n_colors=len(conf_counts))
explode = [0.05 if x == 0.95 else 0 for x in conf_counts.index] # On détache le top score
ax2.pie(conf_counts, labels=[f"Score {x}" for x in conf_counts.index], 
        autopct='%1.1f%%', startangle=140, colors=colors, 
        explode=explode, pctdistance=0.85, shadow=False,
        textprops={'fontsize': 12, 'fontweight': 'bold'})

# Transformation en Donut
centre_circle = plt.Circle((0,0), 0.70, fc='white')
ax2.add_artist(centre_circle)
ax2.set_title("B. Indice de Confiance de l'Alignement", fontsize=18, fontweight='bold', pad=20)
plt.text(0, 0, f"Moyenne\n{df_sota['confidence'].mean():.2f}", 
         ha='center', va='center', fontsize=16, fontweight='bold', color='#4a4a4a')

# Finalisation et Sauvegarde
plt.suptitle("Pipeline SOTA : Validation de la Nomenclature et Intégrité des Séquences", 
             fontsize=24, fontweight='bold', y=1.05)

# Ajout d'une légende explicative en bas
plt.figtext(0.5, -0.05, 
            "Note: Le diagnostic utilise l'alignement local de Smith-Waterman avec pénalités de gap affines (-0.5/-0.1).\n"
            "La confiance '0.95' certifie une correspondance canonique stricte prête pour la modélisation de variants.", 
            ha="center", fontsize=12, style='italic', bbox={"facecolor":"orange", "alpha":0.1, "pad":10})

plt.savefig("sota_uniprot_isoform_diagnostic.png", dpi=300, bbox_inches='tight')
plt.show()
print("\n📊 Bilan Finale des isoformes:")
print(df_sota['status'].value_counts())
print("\nDistribution des scores de confiance :")
print(df_sota['confidence'].value_counts().sort_index())
df_sota.to_csv("uniprot_isoform_diagnostic.csv", index=False)

In [ ]:
import pandas as pd
from tqdm.auto import tqdm
import time
import requests

print("📖 Chargement des fichiers:")
df_main = pd.read_parquet("bindingdb_onco_diamond_clean.parquet")
df_diag = pd.read_csv("uniprot_isoform_diagnostic.csv")
# Extraction propre de l'ID complet (ex: Q9Y618-5)
def extract_full_id(uid):
    uid = str(uid).strip()
    if '|' in uid:
        return uid.split('|')[1]   # prend la partie centrale
    return uid
df_diag['sota_id'] = df_diag['best_match_uniprot'].apply(extract_full_id)
# Mapping ID original → ID SOTA (pour NCOR2 : Q9Y618 → Q9Y618-5)
id_map = dict(zip(df_diag['uniprot_primary'], df_diag['sota_id']))
# Téléchargement des séquences en utilisant le bon ID (sota_id)
unique_sota_ids = df_diag['sota_id'].unique()
full_seq_map = {}
print(f"🚀 Téléchargement des séquences pour {len(unique_sota_ids)} IDs uniques:")
for uid in tqdm(unique_sota_ids):
    if pd.isna(uid) or str(uid).strip() in ["N/A", ""]:
        continue
    url = f"https://rest.uniprot.org/uniprotkb/{uid}.fasta"
    try:
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            full_seq_map[uid] = "".join(r.text.splitlines()[1:]).replace("\n", "").strip()
    except:
        continue
    time.sleep(0.08)
# Application des corrections
df_main['uniprot_id_original'] = df_main['uniprot_id'].copy()
df_main['uniprot_id'] = df_main['uniprot_id'].map(id_map).fillna(df_main['uniprot_id'])  # fallback si mapping échoue
# Injection de la séquence correspondant au nouvel ID
df_main['protein_sequence_full'] = df_main['uniprot_id'].map(full_seq_map)
# 5. Ajout du diagnostic
df_final = df_main.merge(
    df_diag[['uniprot_primary', 'status', 'confidence', 'identity_score_%', 'length_diff']],
    left_on='uniprot_id_original',
    right_on='uniprot_primary',
    how='left'
).drop(columns=['uniprot_primary'], errors='ignore')
# Nettoyage
df_final = df_final.dropna(subset=['protein_sequence_full']).copy()
df_final['protein_sequence_full'] = df_final['protein_sequence_full'].str.upper().str.strip()
# Statistiques
print("\n📊 STATISTIQUES FINALES:")
print(f"le nombre total de paires conservées est : {len(df_final):,}")
print(f"le nombre de gènes/protéines uniques est : {df_final['uniprot_id'].nunique()}")
print(f"le taux de récupération de séquences est : {df_final['protein_sequence_full'].notna().mean()*100:.2f}%")
print("\nDistribution des statuts d'isoforme :")
print(df_final['status'].value_counts(dropna=False))
# Sauvegarde du fichier au format parquet
df_final.to_parquet("bindingdb_onco_diamond_sota_fullseq.parquet", index=False, compression='snappy')

In [ ]:
import pandas as pd

# Chargement du fichier bindingDB
file_path = "bindingdb_onco_diamond_sota_fullseq.parquet"
df_final = pd.read_parquet(file_path)
# Affichage des colonnes et types
print(f"\n📋 le nombre total de colonnes est : {len(df_final.columns)}")
print(df_final.dtypes)
# Statistiques sur l'intégrité
print(f"\n📊 le volume total contient : {len(df_final):,} paires")
print(f"🧬 le nombre totale de protéines uniques est : {df_final['uniprot_id'].nunique()}")
print(f"🧪 le nombre totale de ligands uniques est : {df_final['Ligand SMILES'].nunique():,}")
# Test sur un gène qui était un "Fragment"
# On vérifie qu'il possède maintenant une séquence longue
fragments = df_final[df_final['status'] == 'Isoforme explicite']
if not fragments.empty:
    iso_example = fragments.iloc[0]
    print(f"le gène est : {iso_example['gene_symbol']}")
    print(f"l'uniprot ID est : {iso_example['uniprot_id']}")
    print(f"la longueur de la séquence actuelle est: {len(iso_example['protein_sequence_full'])} AA")
    print(f"Voici un extrait de la séquence protéique : {iso_example['protein_sequence_full'][:50]}")
else:
    print("\n Aucun fragment trouvé dans l'échantillon.")
# Aperçu des 5 premières lignes
print("\n👀 Voici un aperçu des données :")
display(df_final[df_final['status'] == 'Isoforme explicite'].head(13))
# Vérification des labels des gènes drivers oncogènic
print("\n🎯 Répartition des gènes drivers oncogènic :")
print(df_final['is_oncogenic'].value_counts())

## **Partie traitement Cosmic - Clinvar**

In [ ]:
import pandas as pd

# chemin spécifié pour le fichier .tsv
cosmic_tsv_path = "/kaggle/input/datasets/rayanchakibidris/cosmic-genomescreen-v103/Cosmic_GenomeScreensMutant_Tsv_v103_GRCh38/Cosmic_GenomeScreensMutant_v103_GRCh38.tsv/Cosmic_GenomeScreensMutant_v103_GRCh38.tsv"  # ← change ici
# on lit seulement les 10 premières lignes
df_cosmic_sample = pd.read_csv(cosmic_tsv_path, sep='\t', nrows=10, low_memory=False)
print("\n les colonnes présentes dans le fichier genome screen v103 :")
print(list(df_cosmic_sample.columns))
print("\n voici un apercu des dix premières lignes :")
display(df_cosmic_sample)
# on calcule le nombre totale de ligne sans charger tout le fichier 
print("\n le nombre totale de lignes :")
!wc -l {cosmic_tsv_path}

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

print("📖 Chargement du fichier Cosmic:")

cosmic_path = "/kaggle/input/datasets/rayanchakibidris/cosmic-genomescreen-v103/Cosmic_GenomeScreensMutant_Tsv_v103_GRCh38/Cosmic_GenomeScreensMutant_v103_GRCh38.tsv/Cosmic_GenomeScreensMutant_v103_GRCh38.tsv"
df_cosmic = pd.read_csv(cosmic_path, sep='\t', low_memory=False,
                        usecols=['GENE_SYMBOL', 'MUTATION_AA', 'HGVSP', 'HGVSC', 'HGVSG',
                                 'GENOMIC_MUTATION_ID', 'MUTATION_SOMATIC_STATUS',
                                 'TRANSCRIPT_ACCESSION', 'MUTATION_DESCRIPTION'])
print(f"le fichier a été chargé avec succès et contient : {len(df_cosmic):,} lignes")
# Filtre missense réaliste mais rigoureux 
print("\n🔬 Filtre missense réaliste mais rigoureux:")

df_cosmic = df_cosmic[
    df_cosmic['MUTATION_AA'].notna() &
    df_cosmic['MUTATION_AA'].str.startswith('p.') &
    (df_cosmic['MUTATION_AA'] != 'p.?') &
    # Accepte les formats courants : p.R175H, p.Arg175His, p.(Arg175His), p.R175_His, etc.
    df_cosmic['MUTATION_AA'].str.contains(r'p\.[A-Za-z]{1,3}\d+[A-Za-z]{1,3}', regex=True) &
    # Exclusions claires
    ~df_cosmic['MUTATION_AA'].str.contains(r'=', na=False) &        # Exclut synonymous
    ~df_cosmic['MUTATION_AA'].str.contains(r'\*', na=False) &       # Exclut nonsense
    ~df_cosmic['MUTATION_AA'].str.contains(r'del|ins|fs|dup', case=False, na=False)  # Exclut indels/frameshifts
].copy()
print(f"Après avoir filtrer sur les variants missense on obtient : {len(df_cosmic):,} mutations")
# on applique le filtre pour ne garder que les variants somatic confirmés
df_cosmic = df_cosmic[
    df_cosmic['MUTATION_DESCRIPTION'].str.contains("missense", case=False, na=False)
].copy()
print(f"Après avoir filtrer sur les variants confirmés somatic on obtient : {len(df_cosmic):,} mutations")
# on fait un mapping sur les gènes de BindingDB
df_binding = pd.read_parquet("/kaggle/input/datasets/jakeadam68/bindingdb-onco-admet/bindingdb_onco_diamond_sota_fullseq.parquet")
your_genes = set(df_binding['gene_symbol'].dropna().unique())
df_cosmic = df_cosmic[df_cosmic['GENE_SYMBOL'].isin(your_genes)].copy()
print(f"Après avoir fait le mapping sur les gènes de BindingDB on obtient : {len(df_cosmic):,} mutations")
# on fait un nettoyage des HGVSp
def clean_hgvsp(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s.startswith('p.'):
        s = s[2:]
    return s
df_cosmic['HGVSp_clean'] = df_cosmic['HGVSP'].apply(clean_hgvsp)
# Sauvegarde intermédiaire du fichier au format parquet
df_cosmic.to_parquet("cosmic_mutations_filtered_confirmed.parquet", index=False, compression='snappy')
print(f"le nombre de mutations totale après filtrage est : {len(df_cosmic):,} mutations")

In [ ]:
import pandas as pd
# 1. On charge un échantillon du fichier cosmic filtré 
df_cosmic = pd.read_parquet("cosmic_mutations_filtered_confirmed.parquet")
print("📊 Analyse de la colonne MUTATION_DESCRIPTION :")
print("-" * 50)
# Affichage des valeurs uniques et de leur fréquence
description_counts = df_cosmic['MUTATION_DESCRIPTION'].value_counts()
print(description_counts)
# Calcul des pourcentages pour le rapport
print("\n📈 Répartition en pourcentages :")
print((description_counts / len(df_cosmic) * 100).round(2).astype(str) + " %")

In [ ]:
import pandas as pd
import re
from tqdm.auto import tqdm

print("📖 Validation physique des mutations:")
# Chargement des données des fichiers BindingDB et Cosmic
df_bdb = pd.read_parquet("/kaggle/input/datasets/jakeadam68/bindingdb-onco-admet/bindingdb_onco_diamond_sota_fullseq.parquet")
df_cosmic = pd.read_parquet("cosmic_mutations_filtered_confirmed.parquet")
print(f"le fichier bindingDB contient : {len(df_bdb):,} paires")
print(f"le fichier cosmic contient : {len(df_cosmic):,} lignes")
# Dictionnaire optimisé : gene → liste de (uniprot_id, sequence)
gene_to_prots = {}
for _, row in df_bdb.drop_duplicates('uniprot_id').iterrows():
    gene = row['gene_symbol']
    if gene not in gene_to_prots:
        gene_to_prots[gene] = []
    gene_to_prots[gene].append((row['uniprot_id'], row['protein_sequence_full']))
# Fonction de parsing HGVSp améliorée
def parse_hgvsp(hgvsp):
    if pd.isna(hgvsp):
        return None
    s = str(hgvsp).strip()
    if s.startswith('p.'):
        s = s[2:]
    match = re.search(r'([A-Za-z]+)(\d+)([A-Za-z]+)', s)
    if match:
        wt, pos, mut = match.groups()
        aa_map = {'Ala':'A','Arg':'R','Asn':'N','Asp':'D','Cys':'C','Gln':'Q','Glu':'E','Gly':'G',
                  'His':'H','Ile':'I','Leu':'L','Lys':'K','Met':'M','Phe':'F','Pro':'P','Ser':'S',
                  'Thr':'T','Trp':'W','Tyr':'Y','Val':'V'}
        return aa_map.get(wt, wt), int(pos), aa_map.get(mut, mut)
    return None
valid_mutations = []

print("🔬 Validation physique des mutations sur les séquences full-length:")
for _, row in tqdm(df_cosmic.iterrows(), total=len(df_cosmic), desc="Validation"):
    gene = row['GENE_SYMBOL']
    parsed = parse_hgvsp(row['HGVSp_clean'])
    if not parsed or gene not in gene_to_prots:
        continue
    wt_aa, pos, mut_aa = parsed
    # On teste sur toutes les isoformes du gène
    for uid, seq in gene_to_prots[gene]:
        if not seq or pos < 1 or pos > len(seq):
            continue
        if seq[pos-1] == wt_aa:
            mutated_seq = seq[:pos-1] + mut_aa + seq[pos:]
            valid_mutations.append({
                'uniprot_id': uid,
                'gene_symbol': gene,
                'wt_sequence': seq,
                'mutated_sequence': mutated_seq,
                'mutation_hgvsp': row['HGVSP'],
                'genomic_id': row['GENOMIC_MUTATION_ID'],
                'original_hgvsc': row.get('HGVSC')
            })
df_valid = pd.DataFrame(valid_mutations)
# Dédoublonnage final au niveau protéine + mutation
df_final_muts = df_valid.drop_duplicates(subset=['uniprot_id', 'mutation_hgvsp']).copy()
print(f"\nValidation terminée:")
print(f"le nombre de mutations cosmic analysées est : {len(df_cosmic):,} mutations")
print(f"le nombre de mutations validées physiquement est : {len(df_valid):,} mutations")
print(f"le nombre de mutations uniques après dédoublonnage protéique : {len(df_final_muts):,} mutations")
# Sauvegarde du fichier au format parquet
df_final_muts.to_parquet("cosmic_mutations_validated.parquet", index=False, compression='snappy')

In [ ]:
import pandas as pd

print("📖 Lecture du fichier cosmic_mutations_validated.parquet:")

df = pd.read_parquet("cosmic_mutations_validated.parquet")
print(f"\nle fichier a été chargé avec succès et contient : {len(df):,} lignes")
# Affichage des colonnes
print("\n📋 Colonnes présentes dans le fichier :")
print(list(df.columns))
# Aperçu des 10 premières lignes
print("\n👀 Aperçu des 10 premières lignes :")
display(df.head(10))
# Statistiques générales
print("\n📊 Statistiques générales :")
print(df.info())
# Top 15 gènes
print("\n📈 Top 15 gènes les plus représentés :")
print(df['gene_symbol'].value_counts().head(15))
# Statistiques sur la longueur des séquences WT et Mutées
print("\n📏 Statistiques sur la longueur des séquences :")
print("la longueur WT :")
print(df['wt_sequence'].str.len().describe())
print("\nla longueur Mutée :")
print(df['mutated_sequence'].str.len().describe())
# Exemple de mutation
print("\n🔍 Exemple de mutation :")
example = df.iloc[0]
print(f"le gène : {example['gene_symbol']}")
print(f"l'uniprot ID : {example['uniprot_id']}")
print(f"la mutation : {example['mutation_hgvsp']}")
print(f"le séquence WT : {example['wt_sequence'][:80]}")
print(f"la séquence Mutée : {example['mutated_sequence'][:80]}")

In [ ]:
import pandas as pd
from tqdm.auto import tqdm
import os

# Configuration des chemins et paramètres
# Note : vérifie si l'extension est .txt ou .txt.gz sur ton Kaggle
clinvar_path = "/kaggle/input/datasets/jakeadam68/clinvar/variant_summary.txt" 
chunk_size = 500_000 
clinvar_cols = [
    'VariationID', 'GeneSymbol', 'GeneID', 'ClinSigSimple', 
    'ClinicalSignificance', 'LastEvaluated', 'Type', 'Name', 
    'Origin', 'Assembly', 'Chromosome', 'Start', 'Stop', 
    'ReferenceAllele', 'AlternateAllele'
]
# Liste des termes cibles pour la pathogénicité
# Le regex 'Pathogenic|Likely pathogenic' couvrira aussi 'Pathogenic/Likely pathogenic'
pathogenic_regex = r'Pathogenic|Likely pathogenic'
print("📥 Début du traitement de ClinVar par chunks:")
filtered_chunks = []
# Lecture et filtrage simultané
try:
    reader = pd.read_csv(
        clinvar_path, 
        sep='\t', 
        chunksize=chunk_size, 
        usecols=clinvar_cols, 
        low_memory=False,
        dtype=str # on force en string pour éviter les erreurs de type dans les colonnes mixtes
    )
    for chunk in tqdm(reader, desc="Traitement ClinVar"):
        # Application des 3 critères de sélection :
        # A. Type : Uniquement les substitutions de nucléotides (équivalent missense ADN)
        # B. Assembly : Uniquement le génome de référence moderne GRCh38
        # C. ClinicalSignificance : Uniquement les variants dangereux validés
        mask = (
            (chunk['Type'] == 'single nucleotide variant') & 
            (chunk['Assembly'] == 'GRCh38') & 
            (chunk['ClinicalSignificance'].str.contains(pathogenic_regex, case=False, na=False))
        )
        # on ne garde que les lignes qui matchent
        filtered_chunks.append(chunk[mask])
    # Fusion des résultats filtrés
    df_clinvar_patho = pd.concat(filtered_chunks, ignore_index=True)
    # Nettoyage final du DataFrame résultant
    # Conversion des colonnes de position en numérique
    df_clinvar_patho['Start'] = pd.to_numeric(df_clinvar_patho['Start'], errors='coerce')
    df_clinvar_patho['Stop'] = pd.to_numeric(df_clinvar_patho['Stop'], errors='coerce')
    print(f"le nombre de variantes SNV/GRCh38 pathogènes est : {len(df_clinvar_patho):,} variantes")
    print(f"le nombre de gènes uniques représentés est : {df_clinvar_patho['GeneSymbol'].nunique():,} gènes")
    # Sauvegarde du fichier au format parquet
    df_clinvar_patho.to_parquet("clinvar_pathogenic_likely.parquet", index=False)

except FileNotFoundError:
    print(f"Une erreur est survenue, Le fichier n'a pas été trouvé au chemin {clinvar_path}")
except Exception as e:
    print(f"Une erreur est survenue : {e}")

In [ ]:
import pandas as pd

print("📖 Lecture du fichier clinvar:")

df_clinvar = pd.read_parquet("/kaggle/input/datasets/jakeadam68/clinvar/clinvar_pathogenic_likely.parquet")
print(f"\nle fichier a été chargé avec succès et contient : {len(df_clinvar):,} lignes")
# Affichage des colonnes
print("\n📋 Colonnes présentes dans le fichier ClinVar filtré :")
print(list(df_clinvar.columns))
# Aperçu des premières lignes
print("\n👀 Aperçu des 10 premières lignes :")
display(df_clinvar.head(10))
# Statistiques rapides
print("\n📊 Affichage des statistiques :")
print(df_clinvar.info())
print("\nDistribution des significations cliniques :")
print(df_clinvar['ClinicalSignificance'].value_counts().head(15))

In [ ]:
import pandas as pd

# Chargement du fichier intermédiaire
input_path = "/kaggle/input/datasets/jakeadam68/clinvar/clinvar_pathogenic_likely.parquet"
df_clinvar = pd.read_parquet(input_path)
print(f"📖 le fichier a été chargé et contient: {len(df_clinvar):,} lignes")
# Définition des critères de sélection stricts
# on utilise isin() pour un match exact, évitant ainsi les "Conflicting" ou "Low penetrance"
strict_patho_terms = [
    'Pathogenic', 
    'Likely pathogenic', 
    'Pathogenic/Likely pathogenic'
]
print(f"🔬 Filtrage strict sur les termes : {strict_patho_terms}")
# Application du filtre
df_clinvar_strict = df_clinvar[df_clinvar['ClinicalSignificance'].isin(strict_patho_terms)].copy()
# Statistiques de contrôle
print("\n📊 Bilan du filtrage strict :")
print(df_clinvar_strict['ClinicalSignificance'].value_counts())
print("-" * 30)
print(f"le nombre totale de lignes avant filtrage est : {len(df_clinvar):,} lignes")
print(f"le nombre totale de lignes après filtrage est : {len(df_clinvar_strict):,}")
print(f"le nombre totale de lignes supprimées (incertaines/bruit) est : {len(df_clinvar) - len(df_clinvar_strict):,}")
print(f"le nombre de gènes uniques restants est : {df_clinvar_strict['GeneSymbol'].nunique()}")
# Sauvegarde du fichier de référence clinique final au format parquet
output_path = "clinvar_snv_pathogenic_strict.parquet"
df_clinvar_strict.to_parquet(output_path, index=False)
print(f"\nle fichier est sauvegardé au chemin : {output_path}")

In [ ]:
import pandas as pd
import re
from tqdm.auto import tqdm

# Dictionnaire de conversion 3-lettres -> 1-lettre
aa_map = {
    'Ala': 'A', 'Arg': 'R', 'Asn': 'N', 'Asp': 'D', 'Cys': 'C', 'Gln': 'Q', 'Glu': 'E', 
    'Gly': 'G', 'His': 'H', 'Ile': 'I', 'Leu': 'L', 'Lys': 'K', 'Met': 'M', 'Phe': 'F', 
    'Pro': 'P', 'Ser': 'S', 'Thr': 'T', 'Trp': 'W', 'Tyr': 'Y', 'Val': 'V'
}
def normalize_to_1letter(text):
    if pd.isna(text): return None
    s = str(text).strip()
    # on cherche spécifiquement la partie qui commence par 'p.'
    # car c'est elle qui définit la mutation au niveau protéine
    if 'p.' in s:
        s = s.split('p.')[-1].replace(')', '') # On prend ce qu'il y a après p.
    # on cherche l'Acide Aminé (1 ou 3 lettres) + Position + Acide Aminé
    match = re.search(r'([A-Z][a-z]{0,2})(\d+)([A-Z][a-z]{0,2})', s)
    if match:
        wt, pos, mut = match.groups()
        wt_1 = aa_map.get(wt, wt) if len(wt) > 1 else wt
        mut_1 = aa_map.get(mut, mut) if len(mut) > 1 else mut
        # Vérification de sécurité : on s'assure que ce sont des AA valides
        valid_aa = "ACDEFGHIKLMNPQRSTVWY"
        if wt_1 in valid_aa and mut_1 in valid_aa:
            return f"{wt_1}{pos}{mut_1}"
    return None
    
print("📥 Chargement des fichiers pour la fusion finale:")
# Chargement du fichier cosmic validé
df_cosmic = pd.read_parquet("cosmic_mutations_validated.parquet")
print(f"Normalisation de {len(df_cosmic):,} mutations COSMIC:")
# on crée la colonne norm_key
df_cosmic['norm_key'] = df_cosmic['mutation_hgvsp'].apply(normalize_to_1letter)
# on crée la clé de jointure (Gene_Mutation) en utilisant norm_key
df_cosmic['join_key'] = df_cosmic['gene_symbol'].str.upper() + "_" + df_cosmic['norm_key'].fillna('')
# Chargement du fichier clinvar strict
df_clinvar = pd.read_parquet("/kaggle/input/datasets/jakeadam68/clinvar/clinvar_snv_pathogenic_strict.parquet")
print(f"Normalisation de {len(df_clinvar):,} variantes ClinVar:")
# on crée la colonne norm_key dans ClinVar
df_clinvar['norm_key'] = df_clinvar['Name'].apply(normalize_to_1letter)
# On crée la clé de jointure dans ClinVar
df_clinvar['join_key'] = df_clinvar['GeneSymbol'].str.upper() + "_" + df_clinvar['norm_key'].fillna('')
# Déduplication de ClinVar pour éviter de multiplier les lignes COSMIC
df_clinvar_unique = df_clinvar.dropna(subset=['norm_key']).drop_duplicates(subset=['join_key'])
# 4. Fusion des fichiers clinvar et cosmic sur la clé de jointure
print("🔗 Fusion ClinVar ∩ COSMIC:")
df_gold = pd.merge(
    df_cosmic, 
    df_clinvar_unique[['join_key', 'ClinicalSignificance', 'Origin', 'VariationID']], 
    on='join_key', 
    how='inner'
)
print(f"le nombre de variants doublement validés est : {len(df_gold):,}")
print(f"le nombre de gènes représentés est : {df_gold['gene_symbol'].nunique()}")
# Étiquetage et Sauvegarde
df_gold['is_confirmed_pathogenic'] = True
output_name = "oncogenic_variants_final.parquet"
df_gold.to_parquet(output_name, index=False)
print(f"\nle fichier a été sauvegardé avec succès sous : {output_name}")

In [ ]:
import pandas as pd

# Chargement du fichier 
df_labels_preview = pd.read_parquet("oncogenic_variants_final.parquet")
# Affichage des colonnes
print(f"\n📋 Colonnes ({len(df_labels_preview.columns)}) :")
print(df_labels_preview.columns.tolist())
# Statistiques sur la pathogénicité
print("\n📊 Répartition des labels cliniques :")
print(df_labels_preview['ClinicalSignificance'].value_counts())
# Aperçu des données
print("\n👀 Aperçu des 5 premières lignes :")
pd.set_option('display.max_columns', None)
display(df_labels_preview.head(10))
# Vérification d'un gène majeur 
top_genes = df_labels_preview['gene_symbol'].value_counts().head(10)
print(f"\n🔝 Top 5 gènes les plus mutés dans ce set :\n{top_genes}")

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

# Configuration des cibles stratégiques
target_onco = [
    'EGFR', 'KRAS', 'BRAF', 'PIK3CA', 'ALK', 'MET', 'ERBB2', 
    'KIT', 'RET', 'BRCA1', 'BRCA2', 'JAK2', 'ABL1', 'PTEN', 'APC'
]
# Calcul des statistiques
check_df = df_labels_preview[df_labels_preview['gene_symbol'].isin(target_onco)]
counts = check_df['gene_symbol'].value_counts()
# Création d'un tableau récapitulatif
summary_data = []
for gene in target_onco:
    count = counts.get(gene, 0)
    summary_data.append({
        'Gène': gene,
        'Statut': "✅ Validé" if count > 0 else "❌ Non trouvé",
        'Nombre de mutations pathogènes (uniques)': count,
        'Importance Clinique': "Haute" if gene in ['TP53', 'EGFR', 'KRAS', 'BRAF'] else "Standard"
    })

df_summary = pd.DataFrame(summary_data).sort_values(by='Nombre de mutations pathogènes (uniques)', ascending=False)
display(Markdown(f"### Dashboard de validation clinique"))
# Affichage du tableau principal 
def highlight_patho(val):
    color = 'lightgreen' if '✅' in str(val) else 'white'
    return f'background-color: {color}'
display(df_summary.style.hide(axis='index')
        .applymap(highlight_patho, subset=['Statut'])
        .set_properties(**{'text-align': 'center', 'border': '1px solid black'})
        .set_caption("Tableau 1: Couverture des drivers oncogéniques majeurs"))
# Focus spécifique (Exemple EGFR ou Top Gene)
top_gene = df_summary.iloc[0]['Gène']
top_count = df_summary.iloc[0]['Nombre de mutations pathogènes (uniques)']
display(Markdown(f"### 💡 Focus sur les variants de **{top_gene}**"))
display(Markdown(f"Le gène **{top_gene}** présente **{top_count}** mutations pathogènes uniques validées physiquement sur votre séquence SOTA."))
# Affichage des mutations sous forme de colonnes pour gagner de la place
if top_count > 0:
    muts = df_labels_preview[df_labels_preview['gene_symbol'] == top_gene]['norm_key'].tolist()
    # On affiche par blocs de 5 pour la lisibilité
    for i in range(0, len(muts), 5):
        print(" • " + "  • ".join(muts[i:i+5]))

In [ ]:
import pandas as pd

# Chargement du fichier
df_gold = pd.read_parquet("oncogenic_variants_final.parquet")
print(f"Volume avant déduplication protéique : {len(df_gold):,}")
# Utilisation de la colonne 'norm_key' que nous avons créée (ex: L858R)
# la clé d'unicité absolue est : uniprot_id + norm_key (ex: P00533 + L858R)
# On ne garde qu'une seule ligne par mutation physique réelle
df_gold_unique = df_gold.drop_duplicates(subset=['uniprot_id', 'norm_key']).copy()
# bilan après déduplication des hgvsp
print(f"\n Nettoyage des redondances hgvsp :")
print(f"le nombre de mutations physiques uniques est : {len(df_gold_unique):,} mutations pathogènes")
print(f"le nombre de gènes représentés est : {df_gold_unique['gene_symbol'].nunique()} gènes")
# Vérification du gène EGFR 
if 'EGFR' in df_gold_unique['gene_symbol'].values:
    egfr_count = len(df_gold_unique[df_gold_unique['gene_symbol'] == 'EGFR'])
    print(f"le nombre de mutations EGFR uniques restantes : {egfr_count} mutations pathogènes")
    print("Voici un exemple:", df_gold_unique[df_gold_unique['gene_symbol'] == 'EGFR']['norm_key'].head(5).tolist())
# 4. Sauvegarde du fichier
df_gold_unique.to_parquet("oncogenic_variants_sota.parquet", index=False)

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

# Configuration des cibles stratégiques
target_onco = [
    'EGFR', 'KRAS', 'BRAF', 'PIK3CA', 'ALK', 'MET', 'ERBB2', 
    'KIT', 'RET', 'BRCA1', 'BRCA2', 'JAK2', 'ABL1', 'PTEN', 'APC', 'TP53'
]
# Calcul des statistiques
check_df = df_gold_unique[df_gold_unique['gene_symbol'].isin(target_onco)]
counts = check_df['gene_symbol'].value_counts()
# Création d'un tableau récapitulatif
summary_data = []
for gene in target_onco:
    count = counts.get(gene, 0)
    summary_data.append({
        'Gène': gene,
        'Statut': "✅ Validé" if count > 0 else "❌ Non trouvé",
        'Nombre de mutations pathogènes (uniques)': count,
        'Importance Clinique': "Haute" if gene in ['TP53', 'EGFR', 'KRAS', 'BRAF'] else "Standard"
    })

df_summary = pd.DataFrame(summary_data).sort_values(by='Nombre de mutations pathogènes (uniques)', ascending=False)
display(Markdown(f"### Dashboard de validation clinique"))
# Affichage du tableau principal 
def highlight_patho(val):
    color = 'lightgreen' if '✅' in str(val) else 'white'
    return f'background-color: {color}'
display(df_summary.style.hide(axis='index')
        .applymap(highlight_patho, subset=['Statut'])
        .set_properties(**{'text-align': 'center', 'border': '1px solid black'})
        .set_caption("Tableau 1: Couverture des drivers oncogéniques majeurs"))
# Focus spécifique (Exemple EGFR ou Top Gene)
top_gene = df_summary.iloc[0]['Gène']
top_count = df_summary.iloc[0]['Nombre de mutations pathogènes (uniques)']
display(Markdown(f"### 💡 Focus sur les variants de **{top_gene}**"))
display(Markdown(f"Le gène **{top_gene}** présente **{top_count}** mutations pathogènes uniques validées physiquement sur votre séquence SOTA."))
# Affichage des mutations sous forme de colonnes pour gagner de la place
if top_count > 0:
    muts = df_gold_unique[df_gold_unique['gene_symbol'] == top_gene]['norm_key'].tolist()
    # On affiche par blocs de 5 pour la lisibilité
    for i in range(0, len(muts), 5):
        print(" • " + "  • ".join(muts[i:i+5]))

In [ ]:
import pandas as pd
import gc

print("🔗 Fusion des séquences physiques avec les labels ClinVar Dédoublonnés:")
# Chargement du fichier contenant toutes les mutations validées par Cosmic
df_physical = pd.read_parquet("cosmic_mutations_validated.parquet")
# Chargement du fichier contenant les mutations pathogènes dédupliquées
df_labels = pd.read_parquet("/kaggle/input/datasets/rayanchakibidris/cosmic-genomescreen-v103/oncogenic_variants_sota.parquet")
# on fusionne sur l'ID de la protéine et le nom de la mutation
df_pathogenic_sequences = pd.merge(
    df_physical,
    df_labels[['uniprot_id', 'mutation_hgvsp', 'ClinicalSignificance', 'join_key', 'is_confirmed_pathogenic']],
    on=['uniprot_id', 'mutation_hgvsp'],
    how='inner'
)
# Vérification de l'unicité
df_pathogenic_sequences = df_pathogenic_sequences.drop_duplicates(subset=['uniprot_id', 'mutation_hgvsp'])
print(f"le nombre de séquences mutées d'élite récupérées est : {len(df_pathogenic_sequences):,} séquences mutées")
print(f"le nombre de gènes représentés est : {df_pathogenic_sequences['gene_symbol'].nunique()} gènes")
# Sauvegarde du fichier pour ESM-2
output_file = "pathogenic_sequences_sota_final.parquet"
df_pathogenic_sequences.to_parquet(output_file, index=False, compression='snappy')
print(f"💾 Fichier prêt pour la génération des embeddings ESM-2 : {output_file}")
# Nettoyage mémoire
del df_physical, df_labels
gc.collect()

In [ ]:
import pandas as pd

# Chargement du fichier
file_path = "/kaggle/input/datasets/rayanchakibidris/cosmic-genomescreen-v103/pathogenic_sequences_sota_final.parquet"
df_patho = pd.read_parquet(file_path)
print(f"🔍 Inspection de la bibliothèque des variants pathogènes :")
# Affichage des colonnes que contient le fichier
print(f"\n📋 Liste des {len(df_patho.columns)} colonnes :")
print(df_patho.columns.tolist())
# Affichage des statistiques d'intégrités
print("\n📊 Statistiques de base :")
print(f"le nombre total de variants uniques : {len(df_patho):,} variants")
print(f"le nombre de gènes uniques : {df_patho['gene_symbol'].nunique()} gènes")
print(f"le nombre de UniProt IDs uniques : {df_patho['uniprot_id'].nunique()}")
# Vérification de la répartition OOD (Out-of-Distribution)
if 'is_oncogenic' in df_patho.columns:
    print("\n🎯 Répartition pour le Split OOD :")
    print(df_patho['is_oncogenic'].value_counts().rename({True: 'Drivers (Test/Val)', False: 'Global (Train)'}))
# Aperçu des données (colonnes critiques)
print("\n👀 Aperçu des 5 premières lignes :")
# On limite l'affichage des séquences pour la lisibilité
pd.set_option('display.max_colwidth', 50) 
display(df_patho.head(5))
# Vérification physique d'un échantillon
sample = df_patho.iloc[0]
print(f"\n🧪 Focus sur le premier variant : {sample['gene_symbol']} ({sample['mutation_hgvsp']})")
print(f"La longueur wt est : {len(sample['wt_sequence'])} AA")
print(f"La longueur mt est : {len(sample['mutated_sequence'])} AA")
if len(sample['wt_sequence']) == len(sample['mutated_sequence']):
    print("Les longueurs wt et mt sont parfaitement synchronisées.")

In [ ]:
import pandas as pd
# Vérification du contenu du fichier par sécurité
df_check = pd.read_parquet("/kaggle/input/datasets/rayanchakibidris/cosmic-genomescreen-v103/pathogenic_sequences_sota_final.parquet")
print(f"Longueur moyenne des séquences : {df_check['mutated_sequence'].str.len().max()}")
print(f"Longueur moyenne des séquences : {df_check['mutated_sequence'].str.len().mean():.2f}")
print(f"Longueur minimale trouvée : {df_check['mutated_sequence'].str.len().min()}")